### Create Age column based on DOB column (Age when crime was committed)

In [15]:
import pandas as pd 
import os

In [16]:
notebook_dir = os.getcwd()
data_dir = os.path.join(notebook_dir, '..', 'data')
checkpoint_dir = os.path.join(data_dir, 'checkpoints')

In [17]:
cp11 = pd.read_csv("/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/streamlit-app/checkpoint11_combined_data.csv")

In [18]:
cp11

,Incident #,Date,Type,Location,Arrested,Location Prefix,DOB,Charges,raw_address,latitude,longitude,geocode_confidence,person_id,category,Year,crime_severity
0,18000001.0,2018-01-01 00:01:14,NOISE ORD,3 HARRIMAN ST,Yes,NaN,07/03/1974,A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...,3 HARRIMAN ST,42.718522,-71.148148,10.0,20f72481f083d4756c89b98fd499514c5953a8a4c10253...,Public Disturbances,2018,Non-Serious
1,18000002.0,2018-01-01 00:08:38,LOUD NOISE,1 HARRIMAN ST,Yes,NaN,01/21/1979,A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...,1 HARRIMAN ST FL 2,42.718522,-71.148148,10.0,2d5adb553cad8a22fd72d2e390525997c4963bac1b185e...,Public Disturbances,2018,Non-Serious
2,18000003.0,2018-01-01 00:11:17,ALARM/BURG,16 ALLEN ST,No,MATOS,NaN,NaN,16 ALLEN ST,42.710782,-71.151911,10.0,NaN,Fire and Arson Incidents,2018,Non-Serious
3,18000004.0,2018-01-01 00:14:53,DISORDERLY,11 SUMMER ST,No,NaN,NaN,NaN,11 SUMMER ST,42.711117,-71.153015,10.0,NaN,Public Disturbances,2018,Non-Serious
4,18000005.0,2018-01-01 00:27:36,EXTRA SURVEIL,57 SPRINGFIELD ST,Yes,WARD SIX CLUB,06/26/2002,A&B DOMESTIC NO 209A IN EFFECT,57 SPRINGFIELD ST,42.699340,-71.156938,10.0,41c43f9f4255dee8e2c910e87f2a983e1b13beecf27eac...,Preventive Policing,2018,Non-Serious
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
428522,19030902.0,2019-06-18 22:46:26,DISTURBANCE,280 HAVERHILL ST,No,NaN,NaN,NaN,280 HAVERHILL ST,42.709791,-71.165424,10.0,NaN,Public Disturbances,2019,Non-Serious
428523,19030903.0,2019-06-18 22:49:49,M/V STOP,COMMON ST & HAMPSHIRE ST,No,NaN,NaN,NaN,COMMON ST & HAMPSHIRE ST,42.705821,-71.167853,7.0,NaN,Motor Vehicle Incidents,2019,Non-Serious
428524,19030904.0,2019-06-18 23:38:19,TOW OF M/V,30 MYRTLE CT,No,TRESPASS,NaN,NaN,30 MYRTLE CT,42.714430,-71.169718,10.0,NaN,Motor Vehicle Incidents,2019,Non-Serious
428525,19030905.0,2019-06-18 23:49:49,M/V STOP,FITZ ST & LAWRENCE ST,No,NaN,NaN,NaN,FITZ ST & LAWRENCE ST,42.715152,-71.163393,9.0,NaN,Motor Vehicle Incidents,2019,Non-Serious


In [19]:
# work on a copy
df = cp11.copy()

# Optional, tidy column names and Arrested values
df.columns = df.columns.str.strip()
if 'Arrested' in df.columns:
    df['Arrested'] = df['Arrested'].astype(str).str.strip().str.title()

# Keep originals for debugging
df['_DOB_raw'] = df['DOB'].astype(str)
df['_Date_raw'] = df['Date'].astype(str)

# Parse to datetime
df['DOB_dt'] = pd.to_datetime(df['DOB'], errors='coerce', utc=False)
df['Date_dt'] = pd.to_datetime(df['Date'], errors='coerce', utc=False)

# Summary of parsing results
dob_total = df['DOB'].shape[0]
dob_nonnull = df['DOB'].notna().sum()
dob_failed = df['DOB_dt'].isna() & df['DOB'].notna()
date_failed = df['Date_dt'].isna() & df['Date'].notna()

print("DOB non-null count:", dob_nonnull)
print("DOB failed to parse count:", dob_failed.sum())
print("Date failed to parse count:", date_failed.sum())

# Show a few examples of bad DOB and Date entries
if dob_failed.any():
    print("\nSample problematic DOB values:")
    print(df.loc[dob_failed, '_DOB_raw'].drop_duplicates().head(20).to_list())

if date_failed.any():
    print("\nSample problematic Date values:")
    print(df.loc[date_failed, '_Date_raw'].drop_duplicates().head(20).to_list())

# If you want to enforce the standardized columns back into cp11
# only do this once you are happy with the parse results
cp11['DOB'] = df['DOB_dt']
cp11['Date'] = df['Date_dt']

# Final sanity check
print("\nDtypes after standardization:")
print(cp11[['DOB', 'Date']].dtypes)


DOB non-null count: 5093
DOB failed to parse count: 1
Date failed to parse count: 0

Sample problematic DOB values:
['01/28/1068']

Dtypes after standardization:
DOB     datetime64[ns]
Date    datetime64[ns]
dtype: object


In [20]:
import numpy as np

In [21]:
# mask for rows we (those who were charged, assume charged if DOB is present)
mask_charged = cp11['DOB'].notna()

# compute age at offense date with birthday adjustment
before_birthday = (
    (cp11.loc[mask_charged, 'Date'].dt.month < cp11.loc[mask_charged, 'DOB'].dt.month) |
    (
        (cp11.loc[mask_charged, 'Date'].dt.month == cp11.loc[mask_charged, 'DOB'].dt.month) &
        (cp11.loc[mask_charged, 'Date'].dt.day   <  cp11.loc[mask_charged, 'DOB'].dt.day)
    )
)

In [22]:
print(before_birthday)

0          True
1          True
4          True
17         True
19         True
          ...  
428327    False
428433    False
428478     True
428479    False
428480    False
Length: 5092, dtype: bool


In [23]:
age_years = (
    cp11.loc[mask_charged, 'Date'].dt.year - cp11.loc[mask_charged, 'DOB'].dt.year
    - before_birthday.astype(int)
)

In [24]:
print(age_years)

0         43
1         38
4         15
17        30
19        23
          ..
428327    34
428433    57
428478    26
428479    22
428480    31
Length: 5092, dtype: int64


In [25]:
# Write Age, NaN where not charged per your rule
cp11['Age'] = np.nan
cp11.loc[mask_charged, 'Age'] = age_years

In [26]:
# Quick peek
print(cp11[['DOB','Date','Age']].head(10))
print(cp11['Age'].describe())


         DOB                Date   Age
0 1974-07-03 2018-01-01 00:01:14  43.0
1 1979-01-21 2018-01-01 00:08:38  38.0
2        NaT 2018-01-01 00:11:17   NaN
3        NaT 2018-01-01 00:14:53   NaN
4 2002-06-26 2018-01-01 00:27:36  15.0
5        NaT 2018-01-01 00:28:14   NaN
6        NaT 2018-01-01 00:38:54   NaN
7        NaT 2018-01-01 00:42:16   NaN
8        NaT 2018-01-01 00:44:11   NaN
9        NaT 2018-01-01 01:03:11   NaN
count    5092.000000
mean       33.529654
std        11.153846
min        14.000000
25%        25.000000
50%        32.000000
75%        40.000000
max        84.000000
Name: Age, dtype: float64


In [27]:
valid_age_df = cp11.loc[cp11['Age'].notna(), ['DOB', 'Date', 'Age']]

print(valid_age_df.head(20))   # just a preview

           DOB                Date   Age
0   1974-07-03 2018-01-01 00:01:14  43.0
1   1979-01-21 2018-01-01 00:08:38  38.0
4   2002-06-26 2018-01-01 00:27:36  15.0
17  1987-03-10 2018-01-01 01:41:06  30.0
19  1994-05-27 2018-01-01 02:01:26  23.0
39  1982-06-19 2018-01-01 03:20:31  35.0
40  1990-11-18 2018-01-01 03:20:31  27.0
41  1987-05-29 2018-01-01 03:20:31  30.0
42  1991-08-08 2018-01-01 03:20:31  26.0
57  1978-05-23 2018-01-01 05:05:23  39.0
70  1989-04-26 2018-01-01 06:42:45  28.0
71  1983-12-11 2018-01-01 06:42:45  34.0
94  1994-06-11 2018-01-01 10:29:42  23.0
154 1972-03-01 2018-01-01 19:09:31  45.0
155 1997-05-18 2018-01-01 19:09:31  20.0
173 1968-09-30 2018-01-01 21:24:15  49.0
174 1992-08-13 2018-01-01 21:24:15  25.0
175 1987-07-20 2018-01-01 21:25:05  30.0
176 1979-01-07 2018-01-01 21:25:05  38.0
178 1989-12-01 2018-01-01 21:36:52  28.0


In [28]:
save_path = os.path.join(
    notebook_dir,
    "data",
    "missing_dates_csv",
    "md_checkpoints",
    "checkpoint12_age.csv"
)

cp11.to_csv(save_path, index=False)
print(f"Saved with Age column to: {save_path}")

Saved with Age column to: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data/missing_dates_csv/md_checkpoints/checkpoint12_age.csv
